# Autoencoder de anomalías acústicas

Se entrena **solo con sonido normal** del motor. En operación, el M7 calcula el
error de reconstrucción de cada ventana: si sube, el sonido no se parece a lo
aprendido y eso es una anomalía.

Flujo: audio 16 kHz → log-mel (64 bandas) → ventanas de 5 tramas (320 valores)
→ normalización → autoencoder denso → error cuadrático medio (MSE).

**Datos** (WAV de cualquier frecuencia de muestreo; se pasan a mono y 16 kHz):

| Carpeta | Uso |
|---|---|
| `data/normal/` | Motor sano con un PID bien sintonizado (entrenamiento y umbrales) |
| `data/anomalia/` | Opcional: fallas inducidas, solo para evaluar |

Con `DATASET = 'toycar'` usa **ToyCar** del DCASE 2020 Task 2, el mismo
dataset del benchmark de anomalías de MLPerf Tiny, en `data/ToyCar/`
(`train/` solo normal; `test/` normal y anomalía). Sirve para validar el flujo
completo con datos reales, pero es un carrito de juguete, no tu motor.

Si `data/normal/` está vacía, el notebook genera **audio sintético de
demostración** para que se pueda correr de punta a punta. Los umbrales de la
demo no sirven para el motor real.

Salidas en `models/`: `autoencoder.keras`, `autoencoder_int8.tflite`
(para X-CUBE-AI / TFLite Micro) y `autoencoder_params.h` (parámetros de las
features, normalización y umbrales para el firmware).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
print('GPU:', tf.config.list_physical_devices('GPU'))

ROOT = Path('..')
DATA, MODELS = ROOT / 'data', ROOT / 'models'
MODELS.mkdir(exist_ok=True)

# Features: deben ser idénticas en el firmware (CMSIS-DSP)
SR     = 16000   # el M7 diezma de 48 kHz a 16 kHz
N_FFT  = 1024    # 64 ms
HOP    = 512     # 32 ms
N_MELS = 64
FMIN, FMAX = 50, 8000
FRAMES = 5       # tramas por ventana -> entrada de 320 valores
DIM = FRAMES * N_MELS

# Umbrales: percentiles del error en sonido normal, promediado en SMOOTH ventanas
SMOOTH = 8                 # ~0.26 s, para pedir que la anomalía sea sostenida
P_WARN, P_ANOM = 99.0, 99.9

DATASET = 'propio'   # 'propio' (data/normal, data/anomalia) o 'toycar' (data/ToyCar)

SEED = 0
rng = np.random.default_rng(SEED)
tf.keras.utils.set_random_seed(SEED)

## Features: log-mel por ventanas

In [ ]:
WIN = np.hanning(N_FFT + 1)[:-1].astype(np.float32)   # Hann periódica, como arm_hanning
MEL = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)

def logmel(x):
    # Sin padding ni centrado: así lo calcula el firmware trama por trama
    n = 1 + (len(x) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    power = np.abs(np.fft.rfft(x[idx] * WIN, axis=1)) ** 2
    return 10 * np.log10(power @ MEL.T + 1e-10).astype(np.float32)   # (tramas, N_MELS)

def windows(lm):
    v = np.lib.stride_tricks.sliding_window_view(lm, (FRAMES, N_MELS))[:, 0]
    return v.reshape(-1, DIM)                                         # (ventanas, DIM)

def load(path):
    x, _ = librosa.load(path, sr=SR, mono=True)
    return x

## Datos

Cada archivo se convierte en un arreglo de ventanas. Validación y entrenamiento
se separan **por archivo** (no por ventana) para no medir el error con ventanas
casi iguales a las del entrenamiento.

In [ ]:
def demo_clip(anomaly, seconds=10):
    # Motor: fundamental con armónicos, modulación lenta y ruido de banda ancha
    t = np.arange(int(seconds * SR)) / SR
    f0 = rng.uniform(80, 120)
    x = sum(rng.uniform(0.2, 1) / k * np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 6.3))
            for k in range(1, 8))
    x *= 1 + 0.1 * np.sin(2 * np.pi * rng.uniform(0.5, 2) * t)
    x += 0.3 * rng.standard_normal(t.size)
    if anomaly:   # golpes periódicos (rodamiento) y un chillido agudo
        hits = np.zeros_like(t)
        hits[::int(SR / rng.uniform(8, 20))] = 1
        x += 3 * np.convolve(hits, np.exp(-np.arange(200) / 30) * rng.standard_normal(200), 'same')
        x += 0.4 * np.sin(2 * np.pi * rng.uniform(3000, 5000) * t)
    return (0.1 * x / np.abs(x).max()).astype(np.float32)

def wavs(folder):
    return sorted(p for p in folder.glob('**/*') if p.suffix.lower() in ('.wav', '.flac'))

if DATASET == 'toycar':
    test = DATA / 'ToyCar' / 'test'
    normal_files = wavs(DATA / 'ToyCar' / 'train') + sorted(test.glob('normal_*.wav'))
    anom_files = sorted(test.glob('anomaly_*.wav'))
    assert normal_files, 'falta data/ToyCar (ver README)'
else:
    normal_files, anom_files = wavs(DATA / 'normal'), wavs(DATA / 'anomalia')
DEMO = not normal_files
if DEMO:
    print('data/normal/ vacía: usando audio sintético de demostración')
    normal = [windows(logmel(demo_clip(False))) for _ in range(30)]
    anomal = [windows(logmel(demo_clip(True))) for _ in range(10)]
else:
    normal = [windows(logmel(load(p))) for p in tqdm(normal_files, 'normal')]
    anomal = [windows(logmel(load(p))) for p in tqdm(anom_files, 'anomalía')]

order = rng.permutation(len(normal))
n_val = max(1, len(normal) // 5)
val_clips   = [normal[i] for i in order[:n_val]]
train_clips = [normal[i] for i in order[n_val:]]
X_train, X_val = np.concatenate(train_clips), np.concatenate(val_clips)
print(f'normal: {len(normal)} archivos ({len(X_train)} ventanas de entrenamiento, '
      f'{len(X_val)} de validación); anomalía: {len(anomal)} archivos')

# Normalización por característica, solo con el entrenamiento
MEAN = X_train.mean(0)
STD = X_train.std(0) + 1e-3
norm = lambda X: (X - MEAN) / STD
X_train_n, X_val_n = norm(X_train), norm(X_val)

fig, ax = plt.subplots(1, 2 if anomal else 1, figsize=(12, 3), squeeze=False)
ax[0, 0].imshow(normal[0][:, -N_MELS:].T, origin='lower', aspect='auto'); ax[0, 0].set_title('log-mel normal')
if anomal:
    ax[0, 1].imshow(anomal[0][:, -N_MELS:].T, origin='lower', aspect='auto'); ax[0, 1].set_title('log-mel anomalía')
for a in ax[0]: a.set_xlabel('trama'); a.set_ylabel('banda mel')
plt.show()

## Modelo

Autoencoder denso con cuello de botella de 8: tiene que comprimir el sonido
normal y no le alcanza para reconstruir lo que nunca vio. ~100 mil parámetros
(~120 KB en int8), cabe de sobra en la flash del H745.

Cada capa es Dense → BatchNormalization → ReLU, como la referencia de MLPerf
Tiny. BatchNormalization mantiene acotadas las activaciones; sin ella, al
entrenar muchas épocas crecen tanto que el modelo int8 pierde precisión. Al
convertir a TFLite se fusiona con la capa Dense y no ocupa nada extra.

In [ ]:
def build():
    inp = tf.keras.Input((DIM,))
    x = inp
    for u in (128, 64, 8, 64, 128):
        x = tf.keras.layers.Dense(u)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    out = tf.keras.layers.Dense(DIM)(x)
    return tf.keras.Model(inp, out, name='autoencoder')

build().summary()

In [ ]:
# Se crea aquí para que volver a correr esta celda entrene desde cero
model = build()
model.compile(tf.keras.optimizers.Adam(1e-3), loss='mse')
hist = model.fit(
    X_train_n, X_train_n, validation_data=(X_val_n, X_val_n),
    epochs=100, batch_size=256, shuffle=True, verbose=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)])

plt.figure(figsize=(6, 3))
plt.semilogy(hist.history['loss'], label='entrenamiento')
plt.semilogy(hist.history['val_loss'], label='validación')
plt.xlabel('época'); plt.ylabel('MSE'); plt.legend(); plt.grid(True); plt.show()

## Error de reconstrucción y umbrales

`score` = MSE de cada ventana (lo que calcula el M7), promediado en `SMOOTH`
ventanas. Los umbrales salen de los percentiles del score en la validación
normal. Se envían a la pantalla ×1000 (`audio_warn_m`, `audio_anom_m` en
`common/display_link.h`).

In [ ]:
def scores(predict, clips):
    # Una sola predicción para todos los clips; devuelve el score suavizado de cada clip
    Xn = norm(np.concatenate(clips))
    e = np.mean((predict(Xn) - Xn) ** 2, axis=1)
    parts = np.split(e, np.cumsum([len(c) for c in clips])[:-1])
    return [np.convolve(p, np.ones(SMOOTH) / SMOOTH, 'valid') if len(p) >= SMOOTH else p
            for p in parts]

keras_predict = lambda X: model.predict(X, batch_size=1024, verbose=0)
val_s = scores(keras_predict, val_clips)
s_val = np.concatenate(val_s)
WARN, ANOM = np.percentile(s_val, [P_WARN, P_ANOM])
to_m = lambda v: int(min(round(v * 1000), 65535))
print(f'umbral ADVERTENCIA {WARN:.3f} (audio_warn_m = {to_m(WARN)})')
print(f'umbral ANOMALIA    {ANOM:.3f} (audio_anom_m = {to_m(ANOM)})')

plt.figure(figsize=(8, 3))
bins = np.linspace(0, 3 * ANOM, 80)
plt.hist(s_val, bins, alpha=0.6, density=True, label='normal (validación)')
if anomal:
    anom_s = scores(keras_predict, anomal)
    s_anom = np.concatenate(anom_s)
    plt.hist(np.clip(s_anom, 0, bins[-1]), bins, alpha=0.6, density=True, label='anomalía')
plt.axvline(WARN, color='orange', ls='--', label='advertencia')
plt.axvline(ANOM, color='red', ls='--', label='anomalía')
plt.xlabel('score'); plt.legend(); plt.show()

## Evaluación con anomalías (si hay datos en `data/anomalia/`)

In [ ]:
if anomal:
    y = np.r_[np.zeros(len(s_val)), np.ones(len(s_anom))]
    print(f'AUC ROC por ventana: {roc_auc_score(y, np.r_[s_val, s_anom]):.3f}')
    # Métrica de DCASE / MLPerf Tiny: score medio de cada clip
    yc = np.r_[np.zeros(len(val_s)), np.ones(len(anom_s))]
    print(f'AUC ROC por clip:    {roc_auc_score(yc, [s.mean() for s in val_s + anom_s]):.3f}')
    print(f'falsas alarmas (normal >= advertencia): {np.mean(s_val >= WARN):.2%}')
    print(f'anomalías detectadas (>= advertencia):  {np.mean(s_anom >= WARN):.2%}')
    print(f'anomalías detectadas (>= anomalía):     {np.mean(s_anom >= ANOM):.2%}')
else:
    print('Sin datos de anomalía: solo se calcularon los umbrales.')

## Exportar

- `autoencoder.tflite`: float32 (~400 KB). **Es el recomendado**: el M7 tiene
  FPU y el modelo hace ~100 mil multiplicaciones por ventana (<1 ms a 480 MHz,
  con 32 ms entre ventanas), así que no hace falta cuantizar.
- `autoencoder_int8.tflite`: pesos y activaciones int8 (~120 KB), con entrada
  y salida en float. Con ToyCar su error sube ~16 % respecto a Keras, así que
  los umbrales calculados arriba no le sirven tal cual.

In [ ]:
model.save(MODELS / 'autoencoder.keras')
f32 = tf.lite.TFLiteConverter.from_keras_model(model).convert()
(MODELS / 'autoencoder.tflite').write_bytes(f32)
print(f'autoencoder.tflite: {len(f32) / 1024:.1f} KB')

def representative():
    for i in rng.choice(len(X_train_n), min(500, len(X_train_n)), replace=False):
        yield [X_train_n[i:i + 1]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative
tflite = conv.convert()
(MODELS / 'autoencoder_int8.tflite').write_bytes(tflite)
print(f'autoencoder_int8.tflite: {len(tflite) / 1024:.1f} KB')

interp = tf.lite.Interpreter(model_content=tflite)
def tflite_predict(X):
    interp.resize_tensor_input(0, X.shape); interp.allocate_tensors()
    interp.set_tensor(interp.get_input_details()[0]['index'], X); interp.invoke()
    return interp.get_tensor(interp.get_output_details()[0]['index'])

s_q = np.concatenate(scores(tflite_predict, val_clips[:20]))
s_f = np.concatenate(scores(keras_predict, val_clips[:20]))
print(f'score medio Keras {s_f.mean():.4f} vs int8 {s_q.mean():.4f} '
      f'(diferencia relativa {abs(s_q.mean() / s_f.mean() - 1):.1%})')

In [ ]:
# Parámetros para el firmware.
# shortcut: el banco de filtros mel no se exporta; el firmware debe generarlo con la
# fórmula de librosa (Slaney, sin htk) o exportar MEL aquí cuando se implemente.
def c_array(name, a):
    vals = ', '.join(f'{v:.6g}f' for v in a)
    return f'static const float {name}[{len(a)}] = {{ {vals} }};\n'

h = f'''/* Generado por pc/model/notebooks/01_autoencoder.ipynb. No editar a mano. */
#ifndef AUTOENCODER_PARAMS_H
#define AUTOENCODER_PARAMS_H

#define AE_SR        {SR}
#define AE_N_FFT     {N_FFT}
#define AE_HOP       {HOP}
#define AE_N_MELS    {N_MELS}
#define AE_FMIN      {FMIN}
#define AE_FMAX      {FMAX}
#define AE_FRAMES    {FRAMES}
#define AE_DIM       {DIM}
#define AE_SMOOTH    {SMOOTH}

/* Umbrales del score (MSE promediado) x1000, como en dl_thresholds_t */
#define AE_WARN_M    {to_m(WARN)}
#define AE_ANOM_M    {to_m(ANOM)}
#define AE_DEMO      {int(DEMO or DATASET != 'propio')}  /* 1 = no entrenado con el motor real ({'sintético' if DEMO else DATASET}) */

/* x_norm = (logmel_dB - AE_MEAN) / AE_STD, ventana = {FRAMES} tramas seguidas (la más vieja primero) */
''' + c_array('AE_MEAN', MEAN) + c_array('AE_STD', STD) + '\n#endif\n'
(MODELS / 'autoencoder_params.h').write_text(h)
print('models/autoencoder_params.h escrito', '(no usar en el motor real)' if DEMO or DATASET != 'propio' else '')